# Assignment 5: Continuous Bag of Words (CBOW)
**Aim:** Build a CBOW model that predicts the **target (middle) word** from its **context words**.

Window size = 2 → context = 2 words before + 2 words after.
`[w-2, w-1, ___, w+1, w+2]` → predict the missing word.

Steps: a. Data preparation  b. Generate training data  c. Train model  d. Output

In [1]:
import re
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.decomposition import PCA

I0000 00:00:1793640454.569026   25605 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793640456.355999   25605 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## a. Data preparation

In [2]:
data = """Deep learning (also known as deep structured learning) is part of a broader family of machine learning methods based on artificial neural networks with representation learning. Learning can be supervised, semi-supervised or unsupervised. Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning, recurrent neural networks, convolutional neural networks and Transformers have been applied to fields including computer vision, speech recognition, natural language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, material inspection and board game programs, where they have produced results comparable to and in some cases surpassing human expert performance."""

# To use a text file instead:
# data = open("your_file.txt").read()

In [3]:
sentences = data.split(".")
clean_sent = []
for sentence in sentences:
    if sentence.strip() == "":
        continue
    sentence = re.sub("[^A-Za-z0-9]+", " ", sentence)          # remove special characters
    sentence = re.sub(r"(?:^| )\w(?:$| )", " ", sentence).strip()  # remove 1-letter words
    clean_sent.append(sentence.lower())
clean_sent

['deep learning also known as deep structured learning is part of broader family of machine learning methods based on artificial neural networks with representation learning',
 'learning can be supervised semi supervised or unsupervised',
 'deep learning architectures such as deep neural networks deep belief networks deep reinforcement learning recurrent neural networks convolutional neural networks and transformers have been applied to fields including computer vision speech recognition natural language processing machine translation bioinformatics drug design medical image analysis climate science material inspection and board game programs where they have produced results comparable to and in some cases surpassing human expert performance']

In [4]:
# Build vocabulary: each unique word gets an integer index (0 kept for padding)
word_to_index = {}
for sent in clean_sent:
    for w in sent.split():
        if w not in word_to_index:
            word_to_index[w] = len(word_to_index) + 1

sequences = [[word_to_index[w] for w in sent.split()] for sent in clean_sent]
index_to_word = {i: w for w, i in word_to_index.items()}
vocab_size = len(word_to_index) + 1
print("Vocabulary size:", vocab_size)
print(sequences)

Vocabulary size: 74
[[1, 2, 3, 4, 5, 1, 6, 2, 7, 8, 9, 10, 11, 9, 12, 2, 13, 14, 15, 16, 17, 18, 19, 20, 2], [2, 21, 22, 23, 24, 23, 25, 26], [1, 2, 27, 28, 5, 1, 17, 18, 1, 29, 18, 1, 30, 2, 31, 17, 18, 32, 17, 18, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 12, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 33, 59, 60, 61, 62, 63, 35, 64, 65, 66, 38, 33, 67, 68, 69, 70, 71, 72, 73]]


## b. Generate training data (context → target)

In [5]:
window = 2
contexts, targets = [], []
for seq in sequences:
    for i in range(window, len(seq) - window):
        context = seq[i - window:i] + seq[i + 1:i + 1 + window]
        contexts.append(context)
        targets.append(seq[i])

X = np.array(contexts)
Y = np.array(targets)
print("X shape:", X.shape, " Y shape:", Y.shape)

for i in range(5):
    print([index_to_word[w] for w in X[i]], "->", index_to_word[Y[i]])

X shape: (87, 4)  Y shape: (87,)
['deep', 'learning', 'known', 'as'] -> also
['learning', 'also', 'as', 'deep'] -> known
['also', 'known', 'deep', 'structured'] -> as
['known', 'as', 'structured', 'learning'] -> deep
['as', 'deep', 'learning', 'is'] -> structured


## c. Train model
Embedding (word → vector) → Average of context vectors → Dense → Softmax over vocabulary

In [6]:
emb_size = 10

model = keras.Sequential([
    keras.Input(shape=(2 * window,)),
    layers.Embedding(input_dim=vocab_size, output_dim=emb_size),
    layers.GlobalAveragePooling1D(),           # mean of context word embeddings
    layers.Dense(256, activation="relu"),
    layers.Dense(512, activation="relu"),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

history = model.fit(X, Y, epochs=80, verbose=0)
print("Final training accuracy:", history.history["accuracy"][-1])

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 4, 10)          │           740 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 10)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │         2,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 74)             │        37,962 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 173,102 (676.18 KB)

 Trainable params: 173,102 (676.18 KB)

 Non-trainable params: 0 (0.00 B)

Final training accuracy: 1.0


In [7]:
plt.plot(history.history["loss"], label="Loss")
plt.plot(history.history["accuracy"], label="Accuracy")
plt.xlabel("Epoch"); plt.legend(); plt.title("CBOW training"); plt.show()

/tmp/ipykernel_25605/1100602750.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("Epoch"); plt.legend(); plt.title("CBOW training"); plt.show()


## d. Output

In [8]:
# Word embeddings visualised in 2D using PCA
embeddings = model.layers[0].get_weights()[0]
reduced = PCA(n_components=2).fit_transform(embeddings)

plt.figure(figsize=(12, 8))
for word, idx in list(word_to_index.items())[:40]:
    x, y = reduced[idx]
    plt.scatter(x, y)
    plt.annotate(word, (x, y))
plt.title("Word embeddings (PCA)"); plt.show()

/tmp/ipykernel_25605/2679276574.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Word embeddings (PCA)"); plt.show()


In [9]:
# Test: give 4 context words, model predicts the middle word
test_sentences = [
    "known as structured learning",
    "transformers have applied to",
    "where they produced results",
    "cases surpassing expert performance",
]
for sent in test_sentences:
    words = sent.split()
    x = np.array([[word_to_index.get(w, 0) for w in words]])
    pred = np.argmax(model.predict(x, verbose=0))
    print(words[:2], "[", index_to_word.get(pred), "]", words[2:])

['known', 'as'] [ deep ] ['structured', 'learning']


['transformers', 'have'] [ been ] ['applied', 'to']
['where', 'they'] [ have ] ['produced', 'results']
['cases', 'surpassing'] [ human ] ['expert', 'performance']
